In [1]:
import pandas as pd
from h3 import h3
import glob
import numpy as np
from pandas import merge
import json
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import shape
from datetime import datetime, timedelta
import seaborn as sns

In [2]:
# Pandas Config
pd.set_option('display.max_columns', None)
pd.options.display.max_rows = 200

# ShareNow Data 
VEHICLE_TRIP = 'C:/Users/jmuir/source/SharedMobilityAnalytics/data/final_dataset.csv'

# TIMEZONE
TIMEZONE = 'Europe/Berlin'


In [3]:
# Read Trips
trips = pd.read_csv(VEHICLE_TRIP)

C:\Users\jmuir\miniconda3\envs\urban-mobility-simulation\lib\site-packages\IPython\core\interactiveshell.py:3169: DtypeWarning: Columns (4,21,22,114) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,


In [4]:
# convert starts and ends to coarser grid coords using h3_to_parent

# Hex 9
trips.rename(columns={"hex_start":"h3_start_9", "hex_end":"h3_end_9"}, inplace = True)

# Hex 8
trips["h3_start_8"] = trips["h3_start_9"].apply(lambda row: h3.h3_to_parent(row))
trips["h3_end_8"] = trips["h3_end_9"].apply(lambda row: h3.h3_to_parent(row))

# Hex 7
trips["h3_start_7"] = trips["h3_start_8"].apply(lambda row: h3.h3_to_parent(row))
trips["h3_end_7"] = trips["h3_end_8"].apply(lambda row: h3.h3_to_parent(row))

Wall time: 0 ns


In [5]:
# Calculate Time Bucket (1h, 2h, 4h, 6h, 12h)

In [6]:
trips["hour_of_day"] = trips["time_start"]//100

In [7]:
# Calculate Buckets
trips["time_bucket_12h"] = trips["hour_of_day"]//12
trips["time_bucket_8h"] = trips["hour_of_day"]//8
trips["time_bucket_6h"] = trips["hour_of_day"]//6
trips["time_bucket_4h"] = trips["hour_of_day"]//4
trips["time_bucket_2h"] = trips["hour_of_day"]//2

In [8]:
# Strip out unnecessary columns

In [9]:
trips = trips[['id',
 'provider',
 'city',
 'vehicleType',
 'model',
 'datetime_start',
 'date_start',
 'time_start',
 'datetime_end',
 'date_end',
 'time_end',
 'longitude_start',
 'latitude_start',
 'longitude_end',
 'latitude_end',
 'fuel_start',
 'fuel_end',
 'distance',
 'duration',
 'price_driving_start',
 'price_driving_end',
 'price_promo_start',
 'price_promo_end',
 'idle_duration',
 'tier',
 'emio',
 'callabike',
 'bmw 2er active tourer',
 'bmw 2er cabrio',
 'bmw x2',
 'mercedes-benz gla',
 'bmw x1',
 'mercedes-benz a-klasse',
 'mini 3-tuerer',
 'mini 5-tuerer',
 'mini cabrio',
 'mini clubman',
 'mini countryman',
 'mercedes-benz b-klasse',
 'bmw i3',
 'smart fortwo 3rd generation',
 'bmw 1er',
 'nextbike',
 'low quality vehicle',
 'medium quality vehicle',
 'premium quality vehicle',
 'month',
 'dayofweek',
 'weekend',
 'week',
 'is_holiday',
 'sharenow_postmerger',
 'elevation_change',
 'ascent',
 'descent',
 'start_d2c',
 'end_d2c',
 'd2c_change',
 'd2c_direction',
 'discrete_count_9',
 'discrete_count_8',
 'discrete_count_7',
 'provider_merged',
 'provider_fleetsize_weekly',
 'h3_start_7',
 'h3_end_7',
 'h3_start_8',
 'h3_end_8',
 'h3_start_9',
 'h3_end_9',
 'hour_of_day',
 'time_bucket_12h',
 'time_bucket_8h',
 'time_bucket_6h',
 'time_bucket_4h',
 'time_bucket_2h']]

In [10]:
# Save Files (and filter data before corona)
trips[trips["date_start"] <= 20200316].to_csv('shared_vehicle_trips_berlin.csv', index=False)

### Calculate Distributions (Possion for arrival, and multinomial for od)

In [11]:
hex_start_column, hex_end_column = "h3_start_7", "h3_end_7"
temporal_column = "time_bucket_4h"
day_of_week_column = "is_holiday"
date_column = "date_start"

trip_data = trips[trips["date_start"] <= 20200316]

##################################
#### First Calculate Arrival Times
##################################

# Count Trips in spatio temporal resolution
tmp_res = trip_data.groupby(
[date_column, day_of_week_column, temporal_column, hex_start_column]).agg({"id": "count"}).reset_index()

arrival_times = tmp_res.groupby([day_of_week_column, temporal_column, hex_start_column]).agg({"id": "mean"}).reset_index()

# Number of potential temporal combinations (day of week, time of day). We need for each combi a distinct lambda.
number_of_temporal_combinations = len(arrival_times[day_of_week_column].unique()) * len(arrival_times[temporal_column].unique())

# Consider only hexagons that have data for each temporal data bucket
hexagon_to_consider = arrival_times.groupby([hex_start_column]).agg({"id": "count"}).reset_index()
hexagon_to_consider = hexagon_to_consider[hexagon_to_consider["id"] == number_of_temporal_combinations]
hexagon_to_consider.drop(columns={"id"}, inplace=True)
hexagon_to_consider = hexagon_to_consider.sort_values(by=[hex_start_column])
hexagon_to_consider["key"] = 1 # For Cross Join Later

# Strip out records for which we don't have complete data (hexagons where #data points < number_of_temporal_combinations)
arrival_times = arrival_times.merge(hexagon_to_consider, on="h3_start_7", how="inner")

####################################################################
#### Second Calculate Destinations Distribution
####################################################################

destination_counts = trip_data.groupby(
[date_column, day_of_week_column, temporal_column, hex_start_column, hex_end_column]).agg({"id": "count"})

# Raw Combinations Date, Holiday, Time Bucket
raw_date = trip_data.groupby([date_column, day_of_week_column]).agg(
{"id": "count"}).reset_index()
raw_time = trip_data.groupby([temporal_column]).agg({"id": "count"}).reset_index()

raw_date["key"] = 1
raw_time["key"] = 1

raw_temporal = raw_date.merge(raw_time, on="key")

# Raw Combinations of Origin and Destination Hexagons
raw_origin = hexagon_to_consider.copy(deep=True)
raw_destination = hexagon_to_consider.copy(deep=True)

raw_origin["key"] = 1
raw_destination["key"] = 1

raw_destination.rename(columns={hex_start_column: hex_end_column}, inplace=True)

raw_spatial = raw_origin.merge(raw_destination, on="key")

# Join Temporal and Spatial raw blank
raw_od = raw_temporal.merge(raw_spatial, on="key")
raw_od.drop(columns={"id_x", "id_y", "key"}, inplace=True)

# Join raw blank (all feasible) Origin Destination combinations with calculated data
od = raw_od.merge(destination_counts,
              on=[date_column, day_of_week_column, temporal_column, hex_start_column, hex_end_column], how="left")

# Using these origin destination samples, we need to derive a distribution function.
# That distribution function is a conditional probability: P(destination | day_of_week, time_of_day, origin)
# In other words: we need to calcuate for each triplet (day_of_week, time_of_day, origin)
# a distribution function as follows:

# We estimate a ***Multinomial Distribution*** for origin destination trip generation
# We need to calculate for each (Day Of Week, Time_Bucket, Origin) a seperate Multinomial Distribution
# Calculate Daily Trips from Region to Region for each time bucket and weekday
trips_per_day_per_region = od.groupby([day_of_week_column, temporal_column, hex_start_column, hex_end_column]).agg(
{"id": "sum"}).reset_index()
trips_per_day_per_region = trips_per_day_per_region.rename(columns={"id": "total_trips"})

# Calculate for each (Day Of Week, Time_Bucket, Origin) Sum_k^K n_{k}
trip_count_overall = trips_per_day_per_region.groupby([day_of_week_column, temporal_column, hex_start_column]).agg(
{"total_trips": "sum"}).reset_index()
trip_count_overall.rename(columns={"total_trips": "sum_total_trips"}, inplace=True)

# Join Counts with Total Sum
trips_per_day_per_region = trips_per_day_per_region.merge(trip_count_overall,
                                                      on=[day_of_week_column, temporal_column, hex_start_column],
                                                      how="inner")

# Calculate p_i for each destination for each (Day Of Week, Time_Bucket, Origin) triplet
trips_per_day_per_region["prob"] = trips_per_day_per_region["total_trips"] / trips_per_day_per_region[
"sum_total_trips"]


# Merge OD Distribution with Arrival Time Values
result = trips_per_day_per_region.merge(arrival_times, on = [day_of_week_column, temporal_column, hex_start_column], how = "left")

result.rename(columns={"prob":"od_multinom_p_est", "id":"o_poisson_est"}, inplace=True)
result.drop(columns={"key"}, inplace=True)

In [12]:
result

,is_holiday,time_bucket_4h,h3_start_7,h3_end_7,total_trips,sum_total_trips,od_multinom_p_est,o_poisson_est
0,False,0,871f18869ffffff,871f18869ffffff,20.0,386.0,0.051813,3.216667
1,False,0,871f18869ffffff,871f1886bffffff,11.0,386.0,0.028497,3.216667
2,False,0,871f18869ffffff,871f18b04ffffff,1.0,386.0,0.002591,3.216667
3,False,0,871f18869ffffff,871f18b06ffffff,0.0,386.0,0.000000,3.216667
4,False,0,871f18869ffffff,871f18b0bffffff,4.0,386.0,0.010363,3.216667
...,...,...,...,...,...,...,...,...
58795,True,5,871f1d4f6ffffff,871f1d4f1ffffff,1.0,164.0,0.006098,27.333333
58796,True,5,871f1d4f6ffffff,871f1d4f2ffffff,29.0,164.0,0.176829,27.333333
58797,True,5,871f1d4f6ffffff,871f1d4f3ffffff,1.0,164.0,0.006098,27.333333
58798,True,5,871f1d4f6ffffff,871f1d4f4ffffff,1.0,164.0,0.006098,27.333333


In [13]:
# How To Sample from that Distribution for a given Triplet
# Convert Probabilities to Numpy Array
sub_set = result[(result["is_holiday"] == False) & (result["time_bucket_4h"] == 0) & (result["h3_start_7"] == "871f18869ffffff")]

prob_array = sub_set["od_multinom_p_est"].to_numpy()

low_soc_vehicle_sample = np.random.multinomial(1, prob_array) # No of vehicles to be charged


idx = low_soc_vehicle_sample.argmax()

sub_set.iloc[idx] # or sub_set.iloc[idx]["h3_end_7"]

is_holiday                     False
time_bucket_4h                     0
h3_start_7           871f18869ffffff
h3_end_7             871f1d49bffffff
total_trips                       21
sum_total_trips                  386
od_multinom_p_est          0.0544041
o_poisson_est                3.21667
Name: 40, dtype: object